# Introduction

This project implements a CNN (Convolutional Neural Network) for the classification of long phrases written by two authors, Kant and Freud (Spanish translation).

Although the classification task was relatively simple, the main objective of this project was to adapt and apply the code presented in the manual to solve similar tasks using locally stored data.

The CNN was implemented using TensorFlow 2.18, although a few modifications were made to adapt the original implementation to both the specific task and TensorFlow 2.21.

The original code was developed by Ganegedara (2022) in the book *Natural Language Processing with TensorFlow 2*.

GitHub repository: https://github.com/thushv89/packt_nlp_tensorflow_2

In [3]:
# Imports
%matplotlib inline

import os
import json
import random
import re

import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split

from tensorflow.keras.preprocessing.text import Tokenizer
import tensorflow.keras.backend as K
import tensorflow.keras.layers as layers
import tensorflow.keras.regularizers as regularizers
from tensorflow.keras.models import Model


# Reproducibility
seed = 54321

random.seed(seed)
np.random.seed(seed)
tf.random.set_seed(seed)

%env TF_FORCE_GPU_ALLOW_GROWTH=true

env: TF_FORCE_GPU_ALLOW_GROWTH=true


# Data wrangling

## Text cleaning and data splitting

In [4]:
# Read data
def read_data(data_dir):
    pages_list = []    
    filenames = []
    print("Reading files...")

    i = 0
    for root, dirs, files in os.walk(data_dir):
        for fi, f in enumerate(files):
            if 'readme' in f.lower():
                continue
            
            i += 1
            #print("." * i, f, end='\r')

            file_path = os.path.join(root, f)
            with open(file_path, encoding='utf-8') as text_file:
                lines = [line.strip() for line in text_file]
                page = ' '.join(lines)
                pages_list.append(page)
                filenames.append(file_path)

    print(f"\nDetected {len(pages_list)} text files.")
    return pages_list, filenames

# Path where the two categories folder are
folder_pages = r'C:\Users\juanm\Jupyter nootebooks\Curso IA\cnn_authorclass_serving\data'


data, filenames = read_data(folder_pages)

# FIles counts
print(f"{sum(len(page.split()) for page in data)} total words.")
print("Example (start):", data[0][:50])
print("Example (end):", data[-1][-50:])

Reading files...

Detected 5189 text files.
2448160 total words.
Example (start): I CARTA SOBRE EL BACHILLERATO[7] 1873 [1941] Por l
Example (end): e agradable a Dios una intención moralmente buena.


In [5]:
# Paths
base_dir = folder_pages
kant_dir = os.path.join(base_dir, "Kant") # Folder name = Kant
freud_dir = os.path.join(base_dir, "Freud") # Folder name = Freud

# Read text files and assign labels
pages = []
labels = []

def read_txts(ruta, etiqueta):
    for filename in sorted(os.listdir(ruta)):
        if filename.lower().endswith(".txt"):
            ruta_completa = os.path.join(ruta, filename)
            with open(ruta_completa, encoding="utf-8") as f:
                text = f.read().strip()
                pages.append(text)
                labels.append(etiqueta)

# Load texts from each author
read_txts(kant_dir, "Kant") # Label = Kant
read_txts(freud_dir, "Freud") # Label = Freud

# Random train-test split
train_pages, test_pages, train_categories, test_categories = train_test_split(
    pages, labels, test_size=0.2, random_state=42
)

# Build dataframes
train_df = pd.DataFrame({'text': train_pages, 'category': train_categories})
test_df = pd.DataFrame({'text': test_pages, 'category': test_categories})

# Display first examples
train_df.head(10)

,text,category
0,"posible averiguarlo. Dado su ritmo, tenían aqu...",Freud
1,Ir. insectos [155] parecen tener el propósito ...,Kant
2,B) PARTE SINTÉTICA\n4. — El mecanismo de place...,Freud
3,extrañeza que nos produce la singular combinac...,Freud
4,"este segundo factor, que no es, ciertamente, e...",Freud
5,e) Algunas ideas obsesivas y su traducción.\nC...,Freud
6,ADVERTENCIA PRELIMINAR\nAcerca de lo peculiar ...,Kant
7,propósito hice a Leonardo de Vinci objeto de u...,Freud
8,207\nTEORÍA Y PRÁCTICA\ncomo niños menores de ...,Kant
9,"padre y cuándo comunica sinceramente, libre de...",Freud


In [6]:
# Words and phrases to remove during preprocessing

words_to_replace = {
    "capítulo",
    "editorial",
    "introducción",
    "Página",
    "freud",
    "kant",
    "www.lectulandia.com",
    "página",
    "prolegomenos"
}

phrases_to_replace = [
    "obras completas",
    "sigmund freud",
    "immanuel kant",
]


def clean_text(text, words=None, frases=None):

    if words is None:
        words = words_to_replace

    if frases is None:
        frases = phrases_to_replace

    # Convert text to lowercase
    text = text.lower()

    # Remove complete phrases
    for frase in frases:
        text = text.replace(frase.lower(), "")

    # Replace special characters
    text = re.sub(r"[«»“”‘’—–…]", " ", text)

    # Normalize common abbreviations
    text = re.sub(r'\bq\b', 'que', text)

    # Remove individual words
    for word in words:
        text = re.sub(rf'\b{re.escape(word)}\b', '', text)

    # Remove numbers and dates
    text = re.sub(r'\b\d+(?:[\.,]\d+)?\b', '', text)

    # Keep only letters and spaces
    text = re.sub(r'[^a-záéíóúüñ\s]', '', text)

    # Remove multiple spaces
    text = re.sub(r'\s+', ' ', text)

    text = text.strip()

    # Remove first and last 10 words
    words_list = text.split()

    # Remove 
    if len(words_list) > 20:
        words_list = words_list[10:-10]

    return " ".join(words_list)

In [7]:
# Apply text cleaning function
train_df['text'] = train_df['text'].apply(lambda x: clean_text(x, words=words_to_replace, frases=phrases_to_replace))
test_df['text'] = test_df['text'].apply(lambda x: clean_text(x, words=words_to_replace, frases=phrases_to_replace))

In [8]:
# Shuffle data
train_df = train_df.sample(frac=1.0, random_state=seed) # frac=1.0 means using 100% of the data
test_df = test_df.sample(frac=1.0, random_state=seed)

In [9]:
# Explicit label mapping
labels_map = {
    "Freud": 0,
    "Kant": 1
}

n_classes = len(labels_map)

print(f"Label -> ID mapping: {labels_map}")

# Convert string labels into numeric labels
train_df["category"] = train_df["category"].map(labels_map)
test_df["category"] = test_df["category"].map(labels_map)

train_df.head(n=10)

Label -> ID mapping: {'Freud': 0, 'Kant': 1}


,text,category
354,en cuanto dicho valor afectivo es hecho desapa...,0
2235,íntima la conexión de los síntomas siendo una ...,0
3040,ligazón conforme a razón con los motivos impul...,1
1453,reducir de nuevo tales síntomas a representaci...,0
2044,de ello que el mantenimiento de ciertas resist...,0
2743,t odos los presentes habrán escuchado con prof...,0
3589,sacerdotes de amon quizá sean insuficientes nu...,0
468,el uso ordinario del lenguaje de las dos clase...,1
1224,hace unos días se apareció usted en mi casa co...,0
2648,bajo las cuales la razón pura se halla inevita...,1


In [10]:
# Split training data into training and validation sets
train_df, valid_df = train_test_split(
    train_df,
    test_size=0.1,
    random_state=seed
)

print(f"Train size: {train_df.shape}")
print(f"Validation size: {valid_df.shape}")

train_df.head()

Train size: (3735, 2)
Validation size: (416, 2)


,text,category
1472,coincidencia de ambas corrientes da con frecue...,0
3732,como condición necesaria a priori tiene que se...,1
1119,para nuestra vida espiritual contribuyó a la f...,0
2100,primeros pasos si suponemos verdadero aquello ...,0
3813,también ciertos factores efectivos y también p...,0


## Tokenization

In [11]:
# Initialize tokenizer and fit only on training data
tokenizer = Tokenizer()

tokenizer.fit_on_texts(
    train_df["text"].tolist()
)

# Vocabulary size
n_vocab = len(tokenizer.word_index) + 1

print(f"Vocabulary size: {n_vocab}")

Vocabulary size: 52113


Looking for the longest sentence

In [12]:
# Compute token length distribution and selected percentiles
train_df["text"].str.split(" ").str.len().describe(percentiles=[0.01, 0.5, 0.99])

count    3735.000000
mean      447.980723
std        92.142118
min       129.000000
1%        187.020000
50%       482.000000
99%       599.660000
max       679.000000
Name: text, dtype: float64

Padding for setences

In [13]:
# Convert text into sequences of token IDs

train_sequences = tokenizer.texts_to_sequences(
    train_df["text"].tolist()
)

valid_sequences = tokenizer.texts_to_sequences(
    valid_df["text"].tolist()
)

test_sequences = tokenizer.texts_to_sequences(
    test_df["text"].tolist()
)

train_labels = train_df["category"].values
valid_labels = valid_df["category"].values
test_labels = test_df["category"].values


# Maximum sequence length selected from the 99th percentile
max_seq_length = 599


# Pad or truncate sequences
preprocessed_train_sequences = tf.keras.preprocessing.sequence.pad_sequences(
    train_sequences,
    maxlen=max_seq_length,
    padding="post",
    truncating="post"
)

preprocessed_valid_sequences = tf.keras.preprocessing.sequence.pad_sequences(
    valid_sequences,
    maxlen=max_seq_length,
    padding="post",
    truncating="post"
)

preprocessed_test_sequences = tf.keras.preprocessing.sequence.pad_sequences(
    test_sequences,
    maxlen=max_seq_length,
    padding="post",
    truncating="post"
)

In [14]:
K.clear_session()

# Input layer using word IDs
word_id_inputs = layers.Input(
    shape=(max_seq_length,),
    dtype="int32"
)

# Embedding layer
embedding_out = layers.Embedding(
    input_dim=n_vocab,
    output_dim=64
)(word_id_inputs)


# Convolutional layers
conv1_1 = layers.Conv1D(
    100,
    kernel_size=3,
    strides=1,
    padding="same",
    activation="relu"
)(embedding_out)

conv1_2 = layers.Conv1D(
    100,
    kernel_size=4,
    strides=1,
    padding="same",
    activation="relu"
)(embedding_out)

conv1_3 = layers.Conv1D(
    100,
    kernel_size=5,
    strides=1,
    padding="same",
    activation="relu"
)(embedding_out)


# Concatenate convolution outputs
conv_out = layers.Concatenate(axis=-1)(
    [conv1_1, conv1_2, conv1_3]
)


# Max pooling
pool_over_time_out = layers.MaxPool1D(
    pool_size=max_seq_length,
    padding="valid"
)(conv_out)


# Flatten
flatten_out = layers.Flatten()(pool_over_time_out)


# Output layer
out = layers.Dense(
    n_classes,
    activation="softmax",
    kernel_regularizer=regularizers.l2(0.001)
)(flatten_out)


# Build model
cnn_model = Model(
    inputs=word_id_inputs,
    outputs=out
)


# Compile model
cnn_model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)

cnn_model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 599)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 599, 64)   │  3,335,232 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 599, 100)  │     19,300 │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_1 (Conv1D)   │ (None, 599, 100)  │     25,700 │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_2 (Conv1D)   │ (None, 599, 100)  │     32,100 │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 599, 300)  │          0 │ conv1d[0][0],     │
│ (Concatenate)       │                   │            │ conv1d_1[0][0],   │
│                     │                   │            │ conv1d_2[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d       │ (None, 1, 300)    │          0 │ concatenate[0][0] │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 300)       │          0 │ max_pooling1d[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 2)         │        602 │ flatten[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 3,412,934 (13.02 MB)

 Trainable params: 3,412,934 (13.02 MB)

 Non-trainable params: 0 (0.00 B)

# Training

As stated previously, the training process was relatively simple because the main objective of this project was to adapt the original code from the manual to work with locally stored data. Therefore, only five epochs were used.

In [15]:
# Learning rate reduction callback
lr_reduce_callback = tf.keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.1,
    patience=3,
    verbose=1,
    mode="auto",
    min_delta=0.0001,
    min_lr=0.000001
)


# Train model
history_cnn_model = cnn_model.fit(
    preprocessed_train_sequences,
    train_labels,
    validation_data=(
        preprocessed_valid_sequences,
        valid_labels
    ),
    batch_size=128,
    epochs=5,
    callbacks=[lr_reduce_callback]
)

Epoch 1/5
30/30 ━━━━━━━━━━━━━━━━━━━━ 9s 256ms/step - accuracy: 0.6969 - loss: 0.5704 - val_accuracy: 0.7644 - val_loss: 0.4634 - learning_rate: 0.0010
Epoch 2/5
30/30 ━━━━━━━━━━━━━━━━━━━━ 7s 235ms/step - accuracy: 0.8988 - loss: 0.2626 - val_accuracy: 0.9663 - val_loss: 0.1295 - learning_rate: 0.0010
Epoch 3/5
30/30 ━━━━━━━━━━━━━━━━━━━━ 7s 247ms/step - accuracy: 0.9831 - loss: 0.0781 - val_accuracy: 0.9832 - val_loss: 0.0629 - learning_rate: 0.0010
Epoch 4/5
30/30 ━━━━━━━━━━━━━━━━━━━━ 7s 242ms/step - accuracy: 0.9957 - loss: 0.0374 - val_accuracy: 0.9904 - val_loss: 0.0444 - learning_rate: 0.0010
Epoch 5/5
30/30 ━━━━━━━━━━━━━━━━━━━━ 7s 243ms/step - accuracy: 0.9979 - loss: 0.0211 - val_accuracy: 0.9928 - val_loss: 0.0334 - learning_rate: 0.0010


In [16]:
# Evaluate model accuracy on the test set
test_results = cnn_model.evaluate(
    preprocessed_test_sequences,
    test_labels,
    return_dict=True
)

test_results

33/33 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.9942 - loss: 0.0338


{'accuracy': 0.9942196607589722, 'loss': 0.03382132202386856}

In [17]:
# Save trained model
cnn_model.save("cnn_model.keras")

In [18]:
# Save tokenizer for inference

tokenizer_json = tokenizer.to_json()

with open("tokenizer.json", "w", encoding="utf-8") as f:
    f.write(tokenizer_json)

print("Tokenizer saved.")

Tokenizer saved.


In [19]:
# Save inference configuration

inference_config = {
    "max_seq_length": max_seq_length,
    "label_map": {
        "0": "Freud",
        "1": "Kant"
    },
    "padding": "post",
    "truncating": "post"
}

with open("inference_config.json", "w", encoding="utf-8") as f:
    json.dump(
        inference_config,
        f,
        ensure_ascii=False,
        indent=4
    )

print("Inference configuration saved.")

Inference configuration saved.


## Export model

In [20]:
# Export model for TensorFlow Serving
export_path = r'C:\Users\juanm\Jupyter nootebooks\Curso IA\cnn_authorclass_serving'

model_name = "kant_freud_model"
model_version = 1

export_path = os.path.join(
    model_name,
    str(model_version)
)

cnn_model.export(export_path)

print(f"SavedModel exported to: {export_path}")

INFO:tensorflow:Assets written to: kant_freud_model\1\assets


INFO:tensorflow:Assets written to: kant_freud_model\1\assets


Saved artifact at 'kant_freud_model\1'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 599), dtype=tf.int32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 2), dtype=tf.float32, name=None)
Captures:
  1456393341840: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1456373348304: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1456395228496: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1456395228304: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1456395228688: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1456395227536: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1456395227920: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1456395229456: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1456395227152: TensorSpec(shape=(), dtype=tf.resource, name=None)
SavedModel exported to: kant_freud_model\1


## Test save model

In [21]:
# Test the exported SavedModel

loaded_saved_model = tf.saved_model.load(export_path)

print("Available signatures:")
print(list(loaded_saved_model.signatures.keys()))

Available signatures:
['serve', 'serving_default']


## Test inferencia

In [22]:
# Test inference using the exported SavedModel

serving_fn = loaded_saved_model.signatures["serve"]

sample_input = tf.constant(
    preprocessed_test_sequences[:2],
    dtype=tf.int32
)

sample_output = serving_fn(sample_input)

sample_output

{'output_0': <tf.Tensor: shape=(2, 2), dtype=float32, numpy=
 array([[9.9536359e-01, 4.6364269e-03],
        [9.9941516e-01, 5.8486586e-04]], dtype=float32)>}

In [23]:
keras_predictions = cnn_model.predict(
    preprocessed_test_sequences[:2]
)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 92ms/step


In [24]:
savedmodel_predictions = sample_output[
    list(sample_output.keys())[0]
].numpy()

print("Keras predictions:")
print(keras_predictions)

print("\nSavedModel predictions:")
print(savedmodel_predictions)

Keras predictions:
[[9.9536359e-01 4.6364269e-03]
 [9.9941516e-01 5.8486586e-04]]

SavedModel predictions:
[[9.9536359e-01 4.6364269e-03]
 [9.9941516e-01 5.8486586e-04]]


In [25]:
np.testing.assert_allclose(
    keras_predictions,
    savedmodel_predictions,
    rtol=1e-5,
    atol=1e-5
)

print("Keras and SavedModel predictions match.")

Keras and SavedModel predictions match.


# Csv and xslx file with clasifications

In [26]:
# Load model if the kernel was restarted
cnn_model = tf.keras.models.load_model("cnn_model.keras")

In [27]:
# Generate predictions on the test set
test_predictions = cnn_model.predict(preprocessed_test_sequences)

33/33 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step


## Correct and incorrect predictions

In [28]:
# Get predicted classes from softmax outputs
predicted_labels = np.argmax(test_predictions, axis=1)

# argmax returns the index of the highest value
# in an array or tensor

# Boolean masks for correct and incorrect predictions
correct_mask = (predicted_labels == test_labels)
incorrect_mask = (predicted_labels != test_labels)

# Count predictions
num_correct = np.sum(correct_mask)
num_incorrect = np.sum(incorrect_mask)

print(f"Correctly classified examples: {num_correct}")
print(f"Incorrectly classified examples: {num_incorrect}")

Correctly classified examples: 1032
Incorrectly classified examples: 6


In [29]:
# Check label encoding
unique_labels = np.unique(test_labels)
print(unique_labels)

[0 1]


In [30]:
# Create a dataframe containing classification results and prediction probabilities

def df_classification_results_with_probs(test_texts, test_labels, test_predictions, label_map, n_samples=10):
    pred_labels = np.argmax(test_predictions, axis=1)
    correct_mask = (pred_labels == test_labels)

    rows = []
    n_classes = len(label_map)
    for text, true_label, pred_label, probs, correct in zip(test_texts, test_labels, pred_labels, test_predictions, correct_mask):
        # Format probabilities with 3 decimals, example: {'Freud': 0.923, 'Kant': 0.077}
        prob_dict = {label_map[i]: f"{probs[i]:.3f}" for i in range(n_classes)}
        rows.append({
            "Text": text,
            "True Label": label_map[true_label],
            "Predicted Label": label_map[pred_label],
            "Correct": correct,
            "Probabilities": prob_dict
        })

    df = pd.DataFrame(rows)

    print("Correct examples:")
    display(df[df["Correct"]].head(n_samples))
    
    print("Incorrect examples:")
    display(df[~df["Correct"]].head(n_samples))

    return df # return dataframe for further analysis and saving

In [31]:
# Display classification results and prediction probabilities

results_df = df_classification_results_with_probs(
    test_texts=test_df["text"].tolist(),
    test_labels=test_df["category"].to_numpy(),
    test_predictions=test_predictions,
    label_map={0: "Freud", 1: "Kant"},
)

# save the results in csv format
results_df.to_csv(r'C:\Users\juanm\Jupyter nootebooks\Curso IA\cnn_authorclass_serving\df_clasificaciones.csv', sep = ';', encoding= 'latin-1',index=False)
results_df.to_excel(r'C:\Users\juanm\Jupyter nootebooks\Curso IA\cnn_authorclass_serving\df_clasificaciones.xlsx',index=False)

Correct examples:


,Text,True Label,Predicted Label,Correct,Probabilities
0,la clase médica el ejercicio del análisis esta...,Freud,Freud,True,"{'Freud': '0.995', 'Kant': '0.005'}"
1,que la argumentación de eggers contra el sueño...,Freud,Freud,True,"{'Freud': '0.999', 'Kant': '0.001'}"
2,son muchas las cosas que quedan en la incertid...,Kant,Kant,True,"{'Freud': '0.000', 'Kant': '1.000'}"
3,de reposo se desprende que dicha organización ...,Freud,Freud,True,"{'Freud': '0.999', 'Kant': '0.001'}"
4,la fantasía consciente muestre de nuevo la act...,Freud,Freud,True,"{'Freud': '1.000', 'Kant': '0.000'}"
5,entre el mismo y los objetivos más caros a su ...,Freud,Freud,True,"{'Freud': '0.995', 'Kant': '0.005'}"
6,raramente el incentivo de emprender investigac...,Freud,Freud,True,"{'Freud': '1.000', 'Kant': '0.000'}"
7,dolencia por el procedimiento descrito y esta ...,Freud,Freud,True,"{'Freud': '0.999', 'Kant': '0.001'}"
8,de una conexión tal de las cosas en sí mismas ...,Kant,Kant,True,"{'Freud': '0.001', 'Kant': '0.999'}"
9,juicio más reservados y precavidos que hasta a...,Freud,Freud,True,"{'Freud': '0.998', 'Kant': '0.002'}"


Incorrect examples:


,Text,True Label,Predicted Label,Correct,Probabilities
54,inclinación natural innata y pasiones de la in...,Kant,Freud,False,"{'Freud': '0.601', 'Kant': '0.399'}"
173,marido tal un rostro así no es caricatura pues...,Kant,Freud,False,"{'Freud': '0.520', 'Kant': '0.480'}"
646,egipcios griegos sirios romanos y finalmente t...,Freud,Kant,False,"{'Freud': '0.325', 'Kant': '0.675'}"
655,tando por las causas se terminará convirtiendo...,Kant,Freud,False,"{'Freud': '0.569', 'Kant': '0.431'}"
989,m ble se sientan alegres ala mesa difícilmente...,Kant,Freud,False,"{'Freud': '0.922', 'Kant': '0.078'}"
1025,to la manipulación ni la exposición pública el...,Kant,Freud,False,"{'Freud': '0.611', 'Kant': '0.389'}"


## Extra tests

In [32]:
print("Test samples:", len(test_labels))
print("Predictions:", len(predicted_labels))

print("Unique true labels:", np.unique(test_labels, return_counts=True))
print("Unique predicted labels:", np.unique(predicted_labels, return_counts=True))

Test samples: 1038
Predictions: 1038
Unique true labels: (array([0, 1]), array([707, 331]))
Unique predicted labels: (array([0, 1]), array([711, 327]))


In [33]:
train_texts = set(train_df["text"])
test_texts = set(test_df["text"])

overlap = train_texts.intersection(test_texts)

print("Train/Test exact duplicates:", len(overlap))

Train/Test exact duplicates: 0


In [34]:
text_kant = """
La razón busca establecer principios universales mediante los cuales
el conocimiento pueda distinguirse de la mera experiencia sensible.
La libertad no consiste simplemente en actuar según los deseos,
sino en someter la voluntad a una ley que la razón pueda reconocer
como universal.
"""

In [35]:
text_freud = """
El deseo reprimido puede retornar de manera indirecta y manifestarse
en los sueños, los síntomas y otros actos que escapan al control
consciente del sujeto. El conflicto entre la vida psíquica consciente
y aquello que ha sido reprimido puede producir diversas formaciones
del inconsciente.
"""

In [36]:
text_mixed = """
La razón pretende gobernar la conducta mediante principios universales,
pero detrás de esa aparente autonomía de la voluntad pueden encontrarse
deseos inconscientes que determinan silenciosamente nuestras decisiones.
La conciencia cree actuar libremente, aunque una parte de la vida
psíquica permanece fuera de su conocimiento.
"""

In [37]:
text_ambiguous = """
El sujeto busca comprender los principios que orientan su conducta,
pero no siempre puede conocer las causas que determinan sus acciones.
Aquello que creemos elegir conscientemente puede estar condicionado
por procesos que no reconocemos de manera inmediata.
"""

In [38]:
text_ridiculo = """
Me comrpo una garompa y todo me importa muy poco. El otro dia comi una uva y no me gusto, me parece
que me dio nausaeas. Pero bueno me gusta el jamon. La Wikipedia en español es la edición en español o castellano de Wikipedia. 
Al igual que las versiones existentes de Wikipedia en otros idiomas, 
es una enciclopedia de contenido libre, publicada en Internet bajo las licencias libres CC BY-SA 4.0 y GFDL. 
En la actualidad cuenta con 2 138 114 artículos, y es escrita por usuarios voluntarios, es decir, que cualquiera puede editar 
un artículo, corregirlo o ampliarlo. Los servidores son administrados por la Fundación Wikimedia, una organización sin ánimo de lucro cuya 
financiación se basa fundamentalmente en donaciones.
Comenzó el 20 de mayo de 2001, cuatro meses después de lanzarse la edición original en inglés y tras el anuncio de Jimmy Wales de internacionalizar 
el proyecto. Es una de las diez Wikipedias con más artículos de entre todos los idiomas.
"""

In [39]:
texto_corto_freud = """
El inconsciente esta estructurado como un lenguaje
"""

In [40]:
texto_corto_kant = """
La razon no aveces no alcanza
"""

In [41]:
texto_corto_ambiguo = """
No tengo la menor idea que escribir aca
"""

In [42]:
texto_corto_kant = """
La razón práctica determina la voluntad mediante principios universales
"""

In [43]:
texto_mezclado = """
La razón determina la voluntad, pero el deseo inconsciente condiciona la conducta
"""

In [44]:
texto_razon = """
La razón determina la voluntad mediante principios universales.
"""

In [45]:
texto_deseo = """
El deseo inconsciente determina la conducta mediante procesos psíquicos.
"""

In [46]:
texto_mezcla_2 = """
La razón determina la voluntad, pero el deseo inconsciente condiciona la conducta.
"""

In [47]:
texto_freud_razon = """
La razón se convierte en una enemiga que nos priva de tantas posibilidades
de placer. Descubrimos cuánto placer procura escapar a ella por lo menos
temporalmente y entregarse a las seducciones de lo insensato.
"""

In [48]:
texto_freud_razon_largo = """
La razón se convierte en una enemiga que nos priva de tantas posibilidades
de placer. Descubrimos cuánto placer procura escapar a ella por lo menos
temporalmente y entregarse a las seducciones de lo insensato. El deseo
inconsciente encuentra así caminos para expresarse y busca satisfacción
a través de pensamientos, sueños y actos que no son plenamente conscientes.
"""

In [49]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

def predict_text(text):
    cleaned = clean_text(text)

    sequence = tokenizer.texts_to_sequences([cleaned])

    padded = pad_sequences(
        sequence,
        maxlen=max_seq_length,
        padding="post",
        truncating="post"
    )

    probabilities = cnn_model.predict(padded, verbose=0)[0]

    print(f"Freud: {probabilities[0]:.4f}")
    print(f"Kant:  {probabilities[1]:.4f}")
    print(f"Predicción: {'Freud' if np.argmax(probabilities) == 0 else 'Kant'}")

In [50]:
predict_text(texto_freud_razon_largo)

Freud: 0.9150
Kant:  0.0850
Predicción: Freud
